In [11]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import kagglehub


In [12]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
X, y = make_moons(n_samples=10000, noise=0.4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [13]:
from sklearn.model_selection import GridSearchCV
model = DecisionTreeClassifier(random_state = 42)   
model.fit(X_train, y_train)
param_grid = [{
    "max_depth": [1,2,3],
    "max_leaf_nodes" : [3,4,5,6,7],
    "min_samples_leaf":[1,3,4]
    
}]

In [14]:
gridsearch = GridSearchCV(model, param_grid, cv = 3, scoring = "accuracy")

In [15]:
gridsearch.fit(X_train,y_train)
gridsearch.best_params_   #finding the best parameters for this dataset to then use in the random forest

{'max_depth': 2, 'max_leaf_nodes': 4, 'min_samples_leaf': 1}

In [16]:
gridsearch.score(X_test, y_test)

0.863

In [17]:
from sklearn.model_selection import ShuffleSplit   
subsets = []     #split training into 1000 sets of 100, training 100 models
decisions = [[] for _ in range(1000)]
rs = ShuffleSplit( n_splits = 1000, train_size = 100, random_state = 42)

for train_index, _ in rs.split(X_train):
    subsets.append([X_train[train_index], y_train[train_index]])
    

for i in range(1000):
    tree_clf = DecisionTreeClassifier(max_depth= 2, max_leaf_nodes= 4, min_samples_leaf= 1, random_state = i)
    tree_clf.fit(subsets[i][0],subsets[i][1])    #fitting each model and having them predict on the test data
    decisions[i]= tree_clf.predict(X_test)



In [18]:
from scipy.stats import mode
decisions = np.array(decisions)

def accuracy_score(set1,set2):
    result = 0.0
    for i in range(len(set1)):
        if set1[i] == set2[i]:
            result += 1
    return(100*result/len(set1))
         #By getting 1000 decision trees to make a decision and taking the majority, 
            #overfitting to specific quarks is not as big of an issue
majority_vote = mode(decisions, axis = 0, keepdims = False).mode
print(accuracy_score(majority_vote, y_test))

86.9
